# Creating datasets

A dataset is a file with an extension `.mglyph`. It is a ZIP archive with this structure:

```text
dataset.mglyph
├── manifest.json
├── 0000.png
├── 0001.png
└── ...
```

`manifest.json` has this structure:

```json
{
  "name": "Simple Star",
  "creation_time": "2026-04-15T10:30:00",
  "samples": {
    "0": [
      {
        "x": 12.34,
        "filename": "0000.png",
        "metadata": {}
      }
    ],
    "1": [
      {
        "x": 56.78,
        "filename": "0001.png",
        "metadata": {}
      }
    ]
  }
}
```

Where:
- `samples` maps split IDs to lists of samples (`"0"` and `"1"` in this notebook, but these can be literally anything).
- Each sample points to a PNG filename inside the same archive.
- `x` is the regression label for that image.

In [ ]:
import random
from pathlib import Path
from zipfile import ZipFile

import mglyph as mg
import numpy as np

from mglyph_ml.dataset.color import hsl_to_hex
from mglyph_ml.dataset.export import Drawer, export_dataset

# Simple Star

This cell generates a dataset containing a simple star.

In [ ]:
def star(x: float, canvas: mg.Canvas, bordercolor, fillcolor, linewidth) -> None:
    canvas.tr.translate(0, mg.lerp(x, 0, 0.05))
    radius = mg.lerp(x, 0.15, canvas.ysize / 2 * 0.9)

    vertices = []
    for segment in range(5):
        vertices.append(mg.orbit(canvas.center, segment * 2 * np.pi / 5, radius))
        vertices.append(
            mg.orbit(
                canvas.center,
                (segment + 0.5) * 2 * np.pi / 5,
                np.cos(2 * np.pi / 5) / np.cos(np.pi / 5) * radius,
            )
        )

    canvas.polygon(vertices, linecap="round", style="fill", color=fillcolor)
    canvas.polygon(vertices, width=linewidth, linecap="round", style="stroke", color=bordercolor)


def simple_star() -> Drawer:
    bordercolor = "navy"
    fillcolor = "white"
    linewidth = "70p"
    return lambda x, canvas: star(x, canvas, bordercolor, fillcolor, linewidth)


mg.show(simple_star(), x=list(np.linspace(0, 100, 10)))  # type: ignore

export_dataset("Simple Star", Path("data/simple-star-20k-dual.mglyph"), simple_star(), n_samples=20_000)

# Varying star

In [ ]:
colors = ["#F1B8B8", "#cfe8fe", "#DDFFDB"]
background_image = Path("data/img/pumpa.jpg")
import cv2
from mglyph.canvas import Raster
from PIL import Image

def random_background_raster(canvas: mg.Canvas, image_path: Path) -> Raster:
    with Image.open(image_path) as image:
        image_np = np.asarray(image.convert("RGB"), dtype=np.uint8)
        src_h, src_w = image_np.shape[:2]

    raster = canvas.make_raster(canvas.top_left, canvas.bottom_right)
    dst_w = raster.raster_width
    dst_h = raster.raster_height
    target_aspect = dst_w / dst_h

    # Build a smaller random crop to enforce zoom-in while preserving target aspect ratio.
    zoom = random.uniform(1.6, 3.2)
    max_crop_w = src_w / zoom
    max_crop_h = src_h / zoom

    if max_crop_w / max_crop_h > target_aspect:
        crop_h = int(max_crop_h)
        crop_w = int(crop_h * target_aspect)
    else:
        crop_w = int(max_crop_w)
        crop_h = int(crop_w / target_aspect)

    crop_w = max(16, min(crop_w, src_w))
    crop_h = max(16, min(crop_h, src_h))

    left = random.randint(0, src_w - crop_w)
    top = random.randint(0, src_h - crop_h)
    patch = image_np[top : top + crop_h, left : left + crop_w]

    # Rotate with reflected borders so the patch remains fully covered (no black corners).
    angle = random.uniform(0.0, 360.0)
    center = (crop_w / 2.0, crop_h / 2.0)
    matrix = cv2.getRotationMatrix2D(center, angle, 1.0)
    rotated_patch = cv2.warpAffine(
        patch,
        matrix,
        (crop_w, crop_h),
        flags=cv2.INTER_CUBIC,
        borderMode=cv2.BORDER_REFLECT_101,
    )

    fitted_patch = cv2.resize(rotated_patch, (dst_w, dst_h), interpolation=cv2.INTER_LANCZOS4)

    raster.array[..., :3] = fitted_patch
    raster.array[..., 3] = 255
    return raster


def varying_star(x: float, canvas: mg.Canvas) -> None:
    # canvas.rect(canvas.top_left, canvas.bottom_right, random.choice(colors))
    canvas.raster(random_background_raster(canvas, background_image))
    canvas.tr.translate(0, mg.lerp(x, 0, 0.05))
    radius = mg.lerp(x, 0.1, canvas.ysize / 2 * 0.85)

    vertices = []
    for segment in range(5):
        vertices.append(mg.orbit(canvas.center, segment * 2 * np.pi / 5, radius))
        vertices.append(
            mg.orbit(
                canvas.center,
                (segment + 0.5) * 2 * np.pi / 5,
                np.cos(2 * np.pi / 5) / np.cos(np.pi / 5) * radius,
            )
        )

    hue = random.randrange(360)
    saturation = random.randrange(100)
    canvas.polygon(vertices, linecap="round", style="fill", color=hsl_to_hex(hue, saturation, 30))

    hue = random.randrange(360)
    saturation = random.randrange(100)
    line_width = random.uniform(0.15, 1.5) * x
    canvas.polygon(
        vertices,
        width=f"{line_width}p",
        linecap="round",
        style="stroke",
        color=hsl_to_hex(hue, saturation, 50),
    )


mg.show(varying_star, x=list(np.linspace(0, 100, 10)))  # type: ignore

export_dataset("Varying Star 1k Dual", Path("data/varying-star-1k-dual.mglyph"), varying_star, n_samples=1000)

# Star Transparent

In [ ]:
from mglyph import CanvasParameters
from mglyph_ml.dataset.export import create_dataset


def transparent_star() -> Drawer:
    bordercolor = "navy"
    fillcolor = "white"
    linewidth = "70p"
    return lambda x, canvas: star(x, canvas, bordercolor, fillcolor, linewidth)


name = "Simple Star Transparent"
dataset_path = Path("data/simple-star-20k-transparent.mglyph")
seed = 69
n_samples = 20_000

# Ensure deterministic generation and remove old file first.
if dataset_path.exists():
    dataset_path.unlink()
    print(f"Removed existing {dataset_path}")

dataset_path.parent.mkdir(parents=True, exist_ok=True)
np_gen = np.random.default_rng(seed)
random.seed(seed)

dataset = create_dataset(name=name)
drawer = transparent_star()

# Keep split strategy identical to export_dataset().
xvalues_train = np_gen.uniform(0.0, 100.0, n_samples - 2)
xvalues_train = np.append(xvalues_train, [0.0, 100.0])
xvalues_train.sort()

xvalues_test = np_gen.uniform(0.0, 100.0, n_samples - 2)
xvalues_test = np.append(xvalues_test, [0.0, 100.0])
xvalues_test.sort()

for x in xvalues_train:
    dataset.add_sample(drawer, float(x), split="0")

for x in xvalues_test:
    dataset.add_sample(drawer, float(x), split="1")

transparent_canvas = CanvasParameters(
    background_color=(0.0, 0.0, 0.0, 0.0),
    canvas_round_corner=False,
)

dataset.export(dataset_path, canvas_parameters=transparent_canvas)
print(f"Generated: {dataset_path}")

In [ ]:
def scaled_letter(x: float, canvas: mg.Canvas, color, character) -> None:
    canvas.text(
        character,
        (0, 0),
        "Arial",
        mg.lerp(x, 0.05, 2.0),
        anchor="center",
        color=color,
        font_weight="bold",
        font_slant="upright",
    )


def random_letter() -> Drawer:
    color = tuple(random.random() for _ in range(3))
    character = random.choice(
        "abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZěščřžýáíéóúťďňĚŠČŘŽÝÁÍÉÓÚŤĎŇ0123456789"
    )
    return lambda x, canvas: scaled_letter(x, canvas, color, character)


for _ in range(3):
    mg.show(random_letter())  # type: ignore

export_dataset(
    name="Random Letter 10k",
    drawer=random_letter(),
    path=Path("data/random-letter-10k-dual.mglyph"),
    seed=69,
    n_samples=10_000,
)